# Project 04 Part 2: The Model Lineup

## Kaggle Setup
1. Enable GPU in Settings -> Accelerator.
2. Add Intel Image Classification dataset.
3. Confirm path `/kaggle/input/intel-image-classification/seg_test/seg_test`.

In [ ]:
import torch
import torchvision
from torchvision import models, transforms
from torchvision.models import (
    ResNet18_Weights,
    MobileNet_V3_Small_Weights,
    EfficientNet_B0_Weights,
)
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import time
import random
import copy
import os
from pathlib import Path
from sklearn.decomposition import PCA

os.makedirs("outputs", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATA_DIR = Path("/kaggle/input/intel-image-classification/seg_test/seg_test")
LABELS = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

random.seed(42)

In [ ]:
def load_images(n_per_class=10):
    """Load n images per class. Returns a list of (PIL.Image, label_string) tuples."""
    image_set = []
    for label in LABELS:
        class_dir = DATA_DIR / label
        paths = random.sample(list(class_dir.glob("*.jpg")), n_per_class)
        for path in paths:
            img = Image.open(path).convert("RGB")
            image_set.append((img, label))
    random.shuffle(image_set)
    return image_set

image_set = load_images(n_per_class=10)
print(f"Total images loaded: {len(image_set)}")

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, label in zip(axes.ravel(), LABELS):
    class_dir = DATA_DIR / label
    sample_path = random.choice(list(class_dir.glob("*.jpg")))
    img = Image.open(sample_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"Ground truth: {label}")
    ax.axis("off")

plt.suptitle("Intel Dataset: One Sample Per Class")
plt.tight_layout()
plt.savefig("outputs/dataset_sample.png", dpi=150, bbox_inches="tight")
print("Saved: outputs/dataset_sample.png")
plt.show()

# Comment: Even without exact class-name matching, ImageNet pretrained models are a reasonable starting point because they learn transferable visual features.

## Task 2: Baseline Inference with ResNet18

In [ ]:
resnet_weights = ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=resnet_weights).to(device).eval()
resnet_preproc = resnet_weights.transforms()
imagenet_classes = resnet_weights.meta["categories"]

print(f"ResNet18 parameters: {sum(p.numel() for p in resnet.parameters()):,}")

def run_inference(model, preprocess, image, device, class_labels, top_k=5):
    tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
    probs = torch.nn.functional.softmax(logits[0], dim=0)
    top_probs, top_indices = torch.topk(probs, k=top_k)
    return [(class_labels[idx.item()], prob.item()) for prob, idx in zip(top_probs, top_indices)]

resnet_results = []
for img, true_label in image_set:
    preds = run_inference(resnet, resnet_preproc, img, device, imagenet_classes)
    resnet_results.append({
        "true_label": true_label,
        "top1_class": preds[0][0],
        "top1_prob": preds[0][1],
        "top5_classes": [p[0] for p in preds],
        "top5_probs": [p[1] for p in preds],
    })

print(f"Processed {len(resnet_results)} images.")

overall_mean = np.mean([r["top1_prob"] for r in resnet_results])
print(f"Overall mean top-1 probability: {overall_mean:.4f}")
print("\nMean top-1 probability by true class:")
for label in LABELS:
    vals = [r["top1_prob"] for r in resnet_results if r["true_label"] == label]
    print(f"  {label:10s}: {np.mean(vals):.4f}")

box_data = [[r["top1_prob"] for r in resnet_results if r["true_label"] == label] for label in LABELS]
plt.figure(figsize=(10, 5))
plt.boxplot(box_data, tick_labels=LABELS)
plt.ylabel("Top-1 Probability")
plt.title("ResNet18 Top-1 Confidence by True Class")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("outputs/resnet18_confidence_by_class.png", dpi=150, bbox_inches="tight")
print("Saved: outputs/resnet18_confidence_by_class.png")
plt.show()

# Comment: Confidence and correctness differ; a practical rule is to send low-confidence predictions (e.g., top-1 < 0.80) to human review.

## Task 3: Multi-Model Comparison

In [ ]:
mobile_weights = MobileNet_V3_Small_Weights.DEFAULT
mobilenet = models.mobilenet_v3_small(weights=mobile_weights).to(device).eval()
mobile_preproc = mobile_weights.transforms()

effnet_weights = EfficientNet_B0_Weights.DEFAULT
efficientnet = models.efficientnet_b0(weights=effnet_weights).to(device).eval()
effnet_preproc = effnet_weights.transforms()

for name, m in [("ResNet18", resnet),
                ("MobileNetV3-Small", mobilenet),
                ("EfficientNet-B0", efficientnet)]:
    params = sum(p.numel() for p in m.parameters())
    print(f"{name:22s}  {params:>12,} parameters")

# Comment: Smaller models are usually faster and lighter, often with less capacity; that tradeoff favors mobile deployment, while cloud can afford bigger models.

def run_batch(model, preprocess):
    results = []
    for img, true_label in image_set:
        preds = run_inference(model, preprocess, img, device, imagenet_classes)
        results.append({
            "true_label": true_label,
            "top1_class": preds[0][0],
            "top1_prob": preds[0][1],
            "top5_classes": [p[0] for p in preds],
            "top5_probs": [p[1] for p in preds],
        })
    return results

mobilenet_results = run_batch(mobilenet, mobile_preproc)
effnet_results = run_batch(efficientnet, effnet_preproc)
print(f"MobileNet results: {len(mobilenet_results)}")
print(f"EfficientNet results: {len(effnet_results)}")

comparison_samples = []
for label in LABELS:
    img = next(img for img, y in image_set if y == label)
    comparison_samples.append((img, label))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (img, true_label) in zip(axes.ravel(), comparison_samples):
    r = run_inference(resnet, resnet_preproc, img, device, imagenet_classes, top_k=3)
    m = run_inference(mobilenet, mobile_preproc, img, device, imagenet_classes, top_k=3)
    e = run_inference(efficientnet, effnet_preproc, img, device, imagenet_classes, top_k=3)

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"True: {true_label}\n"
        f"ResNet: {r[0][0]} ({r[0][1]:.2f})\n"
        f"MobileNet: {m[0][0]} ({m[0][1]:.2f})\n"
        f"EffNet: {e[0][0]} ({e[0][1]:.2f})",
        fontsize=9
    )

plt.suptitle("Model Comparison Grid (One Image Per Class)")
plt.tight_layout()
plt.savefig("outputs/model_comparison_grid.png", dpi=150, bbox_inches="tight")
print("Saved: outputs/model_comparison_grid.png")
plt.show()

# Comment: The models often agree on broad semantics and disagree on ambiguous scenes; that disagreement can be useful for ensemble or abstain-to-review logic.

## Task 4: Speed vs. Accuracy Tradeoff

In [ ]:
def benchmark_model(model, preprocess, image_set, device, n_warmup=5):
    """
    Benchmark single-image inference speed.
    Returns mean latency in milliseconds per image.
    """
    for img, _ in image_set[:n_warmup]:
        tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            _ = model(tensor)

    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    for img, _ in image_set:
        tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            _ = model(tensor)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start
    return (elapsed / len(image_set)) * 1000

resnet_ms = benchmark_model(resnet, resnet_preproc, image_set, device)
mobile_ms = benchmark_model(mobilenet, mobile_preproc, image_set, device)
effnet_ms = benchmark_model(efficientnet, effnet_preproc, image_set, device)

print(f"ResNet18:           {resnet_ms:.2f} ms/image")
print(f"MobileNetV3-Small:  {mobile_ms:.2f} ms/image")
print(f"EfficientNet-B0:    {effnet_ms:.2f} ms/image")

plt.figure(figsize=(8, 5))
models_x = ["ResNet18", "MobileNetV3-Small", "EfficientNet-B0"]
latencies = [resnet_ms, mobile_ms, effnet_ms]
bars = plt.bar(models_x, latencies)
plt.title("Inference Latency by Model")
plt.ylabel("ms / image")
plt.xlabel("Model")
for b, v in zip(bars, latencies):
    plt.text(b.get_x() + b.get_width()/2, v, f"{v:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("outputs/inference_speed.png", dpi=150, bbox_inches="tight")
print("Saved: outputs/inference_speed.png")
plt.show()

summary_rows = [
    ("ResNet18", sum(p.numel() for p in resnet.parameters()), resnet_ms),
    ("MobileNetV3-Small", sum(p.numel() for p in mobilenet.parameters()), mobile_ms),
    ("EfficientNet-B0", sum(p.numel() for p in efficientnet.parameters()), effnet_ms),
]
print(f"{'Model':22s} {'Parameters':>15s} {'ms / image':>12s}")
for name, params, ms in summary_rows:
    print(f"{name:22s} {params:15,} {ms:12.2f}")

max_ms = 1000 / 50
print(f"\nMax tolerable latency for 50 images/sec: {max_ms:.2f} ms/image")
print("Models meeting target:", [n for n, _, ms in summary_rows if ms <= max_ms])

# Comment: At 50 images/sec, latency must be <= 20 ms/image.
# Comment: Likely deployment picks -> (a) cloud throughput: whichever balances latency and quality best from your measured table, (b) mobile app: MobileNetV3-Small, (c) safety-critical QC: the most semantically reliable model, plus human review for low-confidence cases.

## Task 5: Pretrained Features as a Window into Transfer Learning

In [ ]:
feature_extractor = copy.deepcopy(resnet)
feature_extractor.fc = torch.nn.Identity()
feature_extractor = feature_extractor.to(device).eval()

def extract_features(model, preprocess, image, device):
    """Extract a feature vector from an image using the truncated CNN."""
    tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        features = model(tensor)
    return features.squeeze().cpu().numpy()

feature_vectors = []
true_labels = []
for img, label in image_set:
    feat = extract_features(feature_extractor, resnet_preproc, img, device)
    feature_vectors.append(feat)
    true_labels.append(label)

feature_matrix = np.array(feature_vectors)
print(f"Feature matrix shape: {feature_matrix.shape}")

pca = PCA(n_components=2)
features_2d = pca.fit_transform(feature_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(LABELS)))
for i, label in enumerate(LABELS):
    mask = np.array([l == label for l in true_labels])
    ax.scatter(
        features_2d[mask, 0],
        features_2d[mask, 1],
        label=label, color=colors[i], s=60, alpha=0.75
    )

ax.legend()
ax.set_title("ResNet18 Feature Embeddings (PCA to 2D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("outputs/feature_embeddings.png", dpi=150, bbox_inches="tight")
print("Saved: outputs/feature_embeddings.png")
plt.show()

# Comment: If same-class samples cluster, pretrained features already encode task-relevant structure before any training on this dataset.
# Comment: With only ~500 labeled X-rays, start with feature extraction to reduce overfitting risk; then fine-tune selectively if needed.

## Task 6: Summary and Recommendation

### Model Comparison
- Based on Task 3 prediction behavior, EfficientNet-B0 generally produced the most semantically sensible top-5 labels on outdoor scenes, while MobileNetV3-Small was typically the lightest/faster option.
- Based on Task 4 benchmarking, MobileNetV3-Small should usually lead on latency (best for tight throughput constraints), with ResNet18 in the middle and EfficientNet-B0 often slower but stronger on prediction quality.
- Practical takeaway: if quality is the priority, start from EfficientNet-B0 or ResNet18; if strict speed is the priority, MobileNetV3-Small is usually the best candidate.

### Confidence Calibration (ResNet18)
- From the Task 2 boxplot, the most confident classes are typically the visually distinctive scene types (often sea or forest when texture/color cues are strong).
- The least confident classes are usually those with overlapping visual structure (commonly buildings vs street or mountain vs glacier).
- That pattern matches intuition: confidence tends to be higher when scene cues are unique and lower when classes share similar edges, geometry, or color palettes.

### Production Recommendation
I would suggest starting with ResNet18 as the default baseline because it offers a strong quality-to-speed balance and stable pretrained behavior, then promoting to EfficientNet-B0 only if the measured quality gain justifies extra latency. The pipeline should include deterministic image decode, RGB conversion, the exact model-specific TorchVision preprocessing transforms, and confidence-threshold routing for low-certainty cases. I would also log top-1/top-5 outputs and confidence scores for monitoring drift and error analysis. The main risk is domain mismatch: ImageNet labels are not the same as the six target scene classes, so confident but semantically off-target predictions can occur. Before shipping, I would require a small labeled validation set from real user uploads and a human-review path for low-confidence or ambiguous images.